In [23]:
import numpy as np

class Board:

    def __init__(self):
        self.board = np.full((6, 7), ' ', dtype=str)
        self.empty = 42
        self.record = dict()
        self.record[repr(self)] = 1
        self.state = ' '

    def isLegal(self, isX: bool, move: tuple[int, int]):
        r, c = move[0], move[1] - 1
        if r == 0:
            return ((isX and self.board[5, c] == 'X') or (not isX and self.board[5, c] == 'O'))
        if r == 1:
            return self.board[6 - r, c] == ' '
        return (self.board[7 - r, c] != ' ' and self.board[6 - r, c] == ' ')
    
    def playMove(self, icon: str, move: tuple[int, int]):
        r, c = move[0], move[1] - 1
        if r == 0:
            self.board[1:, c] = self.board[:-1, c]
            self.board[0, c] = ' '
            self.empty += 1
        else:
            self.board[6 - r, c] = icon
            self.empty -= 1
        current_state = repr(self)
        if current_state in self.record:
            if self.record[current_state] == 2:
                self.state = "Game ends on a tie!"
            else:
                self.record[current_state] += 1
        else:
            self.record[current_state] = 1  
        self.checkWin('O' if icon == 'X' else 'X')
        self.checkWin(icon)

        if self.empty == 0 and self.state == ' ':
            self.state = "Game ends on a tie!"

    def checkWin(self, icon: str):
        b = (self.board == icon)
        if np.any(b[:, :-3] & b[:, 1:-2] & b[:, 2:-1] & b[:, 3:]):
            self.state = f"Game ends on {icon}'s win!"
            return
        if np.any(b[:-3, :] & b[1:-2, :] & b[2:-1, :] & b[3:, :]):
            self.state = f"Game ends on {icon}'s win!"
            return
        if np.any(b[:-3, :-3] & b[1:-2, 1:-2] & b[2:-1, 2:-1] & b[3:, 3:]):
            self.state = f"Game ends on {icon}'s win!"
            return
        if np.any(b[3:, :-3] & b[2:-1, 1:-2] & b[1:-2, 2:-1] & b[:-3, 3:]):
            self.state = f"Game ends on {icon}'s win!"
            return

    def __str__(self):
        printer = "  -----------------------------\n"
        for i in range(6):
            printer += f"{6 - i} |"
            for j in range(7):
                current = self.board[i, j]
                if current in ('X', 'O'):
                    printer += f" {current} |"
                else:
                    printer += "   |"
            printer += "\n  -----------------------------\n"
        printer += "    1   2   3   4   5   6   7"
        return printer
    
    def __repr__(self):
        return "".join(self.board.ravel())

In [24]:
class Player:
    def __init__(self, isX: bool):
        self.isX = isX

    def getPossibleMoves(self, board):
        if (board.empty == 0):
            return ["tie"]
        moves = []
        for i in range(1, 8):
            if (board.board[5][i - 1] == 'X' and self.isX) or (board.board[5][i - 1] == 'O' and not self.isX):
                moves.append((0, i))
            for j in range(1, 7):
                if board.board[6 - j][i - 1] == ' ':
                    moves.append((j, i))
                    break
        return moves

    def turn(self, board, printer=True):
        
        """ Essa função está aqui apenas de assinatura, 
            a sua implementação vai variar dependendo da subclasse que vem a seguir"""

        raise NotImplementedError("A subclasse deve implementar o método turn!")

    def __str__(self):
        return 'X' if self.isX else 'O'

In [25]:
class HumanPlayer(Player):
    def turn(self, board, printer=True):
        if printer:
            print(f"{self}'s turn")
            print(f"Possible moves: {self.getPossibleMoves(board)}")
            
        while True:
            toParse = input("Enter move coordinates separated by a comma: ").split(",")
            if (board.empty == 0 and len(toParse) == 0 and toParse[0] == "tie"):
                board.state = "Game ends on a tie!"
                break
            if len(toParse) != 2:
                print("Invalid move! Try again.")
                continue
                
            move = tuple((int(toParse[0]), int(toParse[1])))
            
            if move[0] not in range(0, 7) or move[1] not in range(1, 8):
                print("Invalid move! Try again.")
                continue
                
            if board.isLegal(self.isX, move):  
                board.playMove(str(self), move) 
                return move
            else:
                print("Invalid move! Try again.")

In [26]:
import copy
import random 

class MCTSNode:

    def __init__(self, board, move=None, parent=None, constant=1.41, isX=True):
        self.board = copy.deepcopy(board)
        self.move = move
        self.parent = parent
        self.constant = constant
        self.isX = isX
        self.children = []
        self.wins = 0
        self.visits = 0
        self.untriedMoves = MCTSPlayer(isX).getPossibleMoves(self.board)

    def select(self):
        return max(self.children, key=lambda c: (c.wins / c.visits) + self.constant * np.sqrt(np.log(self.visits) / c.visits))
    
    def expand(self):
        move = self.untriedMoves.pop()
        next_board = copy.deepcopy(self.board)
        icon = 'X' if self.isX else 'O'
        next_board.playMove(icon, move)
        child_node = MCTSNode(next_board, move=move, parent=self, constant=self.constant, isX=not self.isX)
        self.children.append(child_node)
        return child_node
    
    def update(self, result):
        self.visits += 1
        if not self.isX: 
            self.wins += result
        else:
            self.wins += (1.0 - result)
        if self.parent:
            self.parent.update(result)

    def is_fully_expanded(self):
        return len(self.untriedMoves) == 0

    def is_terminal(self):
        return self.board.state != ' '

    def rollout(self):
        board_x, board_o = self.to_bitboard(self.board.board)
        current_x_turn = self.isX
        all_cells_mask = 0b111111_111111_111111_111111_111111_111111_111111
        while True:
            moves = self.get_bit_moves(board_x, board_o, current_x_turn)
            if not moves: return 0.5
            move_bit, is_pop = random.choice(moves)
            if is_pop:
                board_x, board_o = self.apply_bit_pop(board_x, board_o, move_bit)
            else:
                if current_x_turn: board_x |= move_bit
                else: board_o |= move_bit
            x_wins = self.check_bit_win(board_x)
            o_wins = self.check_bit_win(board_o)
            if x_wins and o_wins: return 0.5
            if x_wins: return 1.0
            if o_wins: return 0.0
            if (board_x | board_o) == all_cells_mask: return 0.5
            current_x_turn = not current_x_turn

    def to_bitboard(self, grid):
        board_x = 0
        board_o = 0
        for c in range(7):
            for r in range(6):
                shift = c * 7 + r
                if grid[5-r, c] == 'X':
                    board_x |= (1 << shift)
                elif grid[5-r, c] == 'O':
                    board_o |= (1 << shift)
        return board_x, board_o
    
    def get_bit_moves(self, board_x, board_o, isX):
        moves = []
        occupied = board_x | board_o
        for c in range(7):
            bottom_bit = 1 << (c * 7)
            if isX:
                if board_x & bottom_bit: moves.append((bottom_bit, True))
            else:
                if board_o & bottom_bit: moves.append((bottom_bit, True))
            top_bit = 1 << (c * 7 + 5)
            if not (occupied & top_bit):
                column_mask = 0b111111 << (c * 7)
                empty_in_col = (~occupied) & column_mask
                lowest_empty = empty_in_col & -empty_in_col
                moves.append((lowest_empty, False))
        return moves
    
    def apply_bit_pop(self, board_x, board_o, move_bit):
        col_idx = 0
        temp_bit = move_bit
        while temp_bit > 0b111111:
            temp_bit >>= 7
            col_idx += 1
        col_mask = 0b111111 << (col_idx * 7)
        bits_above_mask = (col_mask ^ move_bit) & (-(move_bit << 1))
        new_x = (board_x & ~col_mask) | ((board_x & bits_above_mask) >> 1)
        new_o = (board_o & ~col_mask) | ((board_o & bits_above_mask) >> 1)
        return new_x, new_o

    def check_bit_win(self, bitboard):
        m = bitboard & (bitboard >> 7)
        if m & (m >> 14): return True
        m = bitboard & (bitboard >> 1)
        if m & (m >> 2): return True
        m = bitboard & (bitboard >> 6)
        if m & (m >> 12): return True
        m = bitboard & (bitboard >> 8)
        if m & (m >> 16): return True
        return False

In [27]:
def mcts_base(rootBoard, isX: bool, iterations=10000, constant=1.41):
    
    # variavel responsavel por guardar o estado atual
    rootNode = MCTSNode(rootBoard, None, None, constant, isX)

    for _ in range(iterations):                   
        node = rootNode

        # Seleção
        while node.is_fully_expanded() and not node.is_terminal():
            node = node.select()

        # Expansão
        if not node.is_terminal():
            node = node.expand()
        
        # Simulação
        result = node.rollout()

        # BackPropagation
        node.update(result)

    best_move_node = max(rootNode.children, key=lambda c: c.visits)
    
    return best_move_node.move, best_move_node.move

In [28]:
class MCTSPlayer(Player):
    def __init__(self, isX: bool, type_name="MCTS",mcts_function=mcts_base):
        super().__init__(isX)

        self.type = type_name
        self.mcts_function = mcts_function

    
    def turn(self, board, printer=True):
        if printer:
            print(f"{self}'s turn (MCTS pensando...)")
        
        move = self.mcts_function(board, self.isX, 10000)
        
        board.playMove(str(self), move) 
        return move

In [29]:
class ConfigurableNode(MCTSNode):
    
    def __init__(self, board, move=None, parent=None, constant=1.41, isX=True, 
                 h_win=True, h_block=True, h_safe=True):
        super().__init__(board, move, parent, constant, isX)
        self.h_win = h_win
        self.h_block = h_block
        self.h_safe = h_safe

    def expand(self):
        move = self.untriedMoves.pop()
        next_board = copy.deepcopy(self.board)
        icon = 'X' if self.isX else 'O'
        next_board.playMove(icon, move)
        
        child_node = ConfigurableNode(
            next_board, move=move, parent=self, constant=self.constant, isX=not self.isX,
            h_win=self.h_win, h_block=self.h_block, h_safe=self.h_safe
        )
        self.children.append(child_node)
        return child_node

    # --- FUNÇÕES MODULARIZADAS DE HEURÍSTICA ---

    def get_winning_move(self, moves, board_x, board_o, current_x_turn):
        """Retorna o primeiro movimento que resulta em vitória imediata."""
        for move_bit, is_pop in moves:
            if is_pop:
                next_x, next_o = self.apply_bit_pop(board_x, board_o, move_bit)
            else:
                next_x = (board_x | move_bit) if current_x_turn else board_x
                next_o = board_o if current_x_turn else (board_o | move_bit)
            
            if current_x_turn and self.check_bit_win(next_x):
                return (move_bit, is_pop)
            elif not current_x_turn and self.check_bit_win(next_o):
                return (move_bit, is_pop)
        return None

    def get_blocking_move(self, moves, board_x, board_o, current_x_turn):
        """Encontra se o adversário tem um movimento de vitória e tenta ocupar aquele espaço."""
        enemy_moves = self.get_bit_moves(board_x, board_o, not current_x_turn)
        for enemy_bit, enemy_is_pop in enemy_moves:
            if enemy_is_pop: continue # Bloquear um pop adversário é inviável por colocação direta
            
            sim_x = board_x if current_x_turn else (board_x | enemy_bit)
            sim_o = (board_o | enemy_bit) if current_x_turn else board_o
            
            if (current_x_turn and self.check_bit_win(sim_o)) or (not current_x_turn and self.check_bit_win(sim_x)):
                # Se o inimigo ganha jogando ali, e eu posso jogar ali, eu jogo primeiro!
                if (enemy_bit, False) in moves:
                    return (enemy_bit, False)
        return None

    def get_safe_moves(self, moves, board_x, board_o, current_x_turn):
        """Filtra movimentos que dariam ao adversário a chance de ganhar no próximo turno."""
        safe_moves = []
        for move_bit, is_pop in moves:
            if is_pop:
                my_next_x, my_next_o = self.apply_bit_pop(board_x, board_o, move_bit)
            else:
                my_next_x = (board_x | move_bit) if current_x_turn else board_x
                my_next_o = board_o if current_x_turn else (board_o | move_bit)

            # Simula as respostas do inimigo ao meu movimento
            opponent_wins = False
            enemy_moves = self.get_bit_moves(my_next_x, my_next_o, not current_x_turn)
            for enemy_bit, enemy_is_pop in enemy_moves:
                if enemy_is_pop:
                    sim_x, sim_o = self.apply_bit_pop(my_next_x, my_next_o, enemy_bit)
                else:
                    sim_x = my_next_x if current_x_turn else (my_next_x | enemy_bit)
                    sim_o = (my_next_o | enemy_bit) if current_x_turn else my_next_o

                if (not current_x_turn and self.check_bit_win(sim_x)) or (current_x_turn and self.check_bit_win(sim_o)):
                    opponent_wins = True
                    break

            if not opponent_wins:
                safe_moves.append((move_bit, is_pop))
        
        # Se todos os movimentos são suicidas, retorna todos (não há como escapar, jogue qualquer um)
        return safe_moves if safe_moves else moves

    # --- O NOVO ROLLOUT LIMPO ---

    def rollout(self):
        board_x, board_o = self.to_bitboard(self.board.board)
        current_x_turn = self.isX
        all_cells_mask = 0b111111_111111_111111_111111_111111_111111_111111

        while True:
            moves = self.get_bit_moves(board_x, board_o, current_x_turn)
            if not moves: return 0.5
            
            chosen_move = None
            valid_moves = moves
            
            # Heurística 3: Evitar suicídio (restringe a lista de movimentos possíveis)
            if self.h_safe:
                valid_moves = self.get_safe_moves(valid_moves, board_x, board_o, current_x_turn)

            # Heurística 1: Instinto Assassino
            if self.h_win:
                chosen_move = self.get_winning_move(valid_moves, board_x, board_o, current_x_turn)
            
            # Heurística 2: Defesa (só roda se não achei uma vitória imediata)
            if self.h_block and not chosen_move:
                chosen_move = self.get_blocking_move(valid_moves, board_x, board_o, current_x_turn)
            
            # Se nenhuma heurística ativou (ou estão desligadas), escolhe aleatório dentro dos seguros
            if not chosen_move:
                chosen_move = random.choice(valid_moves)

            # Aplica o movimento e avança
            move_bit, is_pop = chosen_move
            if is_pop:
                board_x, board_o = self.apply_bit_pop(board_x, board_o, move_bit)
            else:
                if current_x_turn: board_x |= move_bit
                else: board_o |= move_bit
                
            x_wins = self.check_bit_win(board_x)
            o_wins = self.check_bit_win(board_o)
            
            if x_wins and o_wins: return 0.5
            if x_wins: return 1.0
            if o_wins: return 0.0
            if (board_x | board_o) == all_cells_mask: return 0.5
            
            current_x_turn = not current_x_turn

In [30]:
def mcts_configuravel(rootBoard, isX: bool, iterations=5000, constant=1.41, h_win=True, h_block=True, h_safe=True):
    
    rootNode = ConfigurableNode(rootBoard, None, None, constant, isX, h_win, h_block, h_safe)

    for _ in range(iterations):                   
        node = rootNode
        while node.is_fully_expanded() and not node.is_terminal():
            node = node.select()
        if not node.is_terminal():
            node = node.expand()
        
        result = node.rollout()
        node.update(result)

    best_move_node = max(rootNode.children, key=lambda c: c.visits)
    return best_move_node.move

In [31]:
print("--- Iniciando: MCTS Vs MCTS_Survival ---")

if random.randint(0,1) == 1:
    playerX = MCTSPlayer(isX=True,mcts_function=mcts_configuravel)
    playerO = MCTSPlayer(isX=False)
else:
    playerO = MCTSPlayer(isX=True,mcts_function=mcts_configuravel)
    playerX = MCTSPlayer(isX=False)

board = Board()
print(board)

while board.state == ' ':
    playerX.turn(board)
    print(board)
    
    if board.state != ' ':
        break
        
    playerO.turn(board)
    print(board)

print(board.state)

--- Iniciando: MCTS Vs MCTS_Survival ---
  -----------------------------
6 |   |   |   |   |   |   |   |
  -----------------------------
5 |   |   |   |   |   |   |   |
  -----------------------------
4 |   |   |   |   |   |   |   |
  -----------------------------
3 |   |   |   |   |   |   |   |
  -----------------------------
2 |   |   |   |   |   |   |   |
  -----------------------------
1 |   |   |   |   |   |   |   |
  -----------------------------
    1   2   3   4   5   6   7
O's turn (MCTS pensando...)


TypeError: unsupported operand type(s) for -: 'tuple' and 'int'

In [25]:
'''
class SurvivalMCTSPlayer(Player):
    def turn(self, board, printer=True):
        if printer:
            print(f"{self}'s turn (SurvivalMCTS pensando...)")
        
        move = mcts_survival(board, self.isX, 10000)
        
        board.playMove(str(self), move[0]) 
        return move[1]
'''

'\nclass SurvivalMCTSPlayer(Player):\n    def turn(self, board, printer=True):\n        if printer:\n            print(f"{self}\'s turn (SurvivalMCTS pensando...)")\n\n        move = mcts_survival(board, self.isX, 10000)\n\n        board.playMove(str(self), move[0]) \n        return move[1]\n'